# (1)Predicting Sodium Content in Fast Food: A Classification Analysis of U.S. Restaurant Menu Items

## Group Final Report — STAT 301 — G4:301P

**Group Members:**
- Aryan Taneja
- Dana Assali
- Sarah Raheja
- Shaam Purushothaman Jayakumaran

In [12]:
library(glmnet)
library(broom)
library(dplyr)
library(ggplot2)
library(car)

Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack


Loaded glmnet 4.1-10



ERROR: Error in library(car): there is no package called ‘car’


## (2)Introduction

Fast food consumption is a major feature of modern North American diets. While it is commonly associated with excess calories and fat, sodium content represents an equally significant public health concern that is often underrepresented in nutritional discussions. Between 2021 and 2023, approximately one in three adults in the United States consumed fast food on a given day, with fast food accounting for roughly 11.7% of daily caloric intake (Shah et al., 2025). High-sodium diets are strongly linked to hypertension, cardiovascular disease, and stroke (Gardener et al., 2012; Wang et al., 2020). According to the U.S. Food and Drug Administration (2024), the average American consumes about 3,400 mg of sodium per day, well above the recommended daily limit of 2,300 mg.

Public health guidelines provide specific benchmarks for reasonable sodium levels in a single serving: 800 mg for sandwiches, salads, and entrées; 480 mg for soups; and 200 mg for snacks and vegetables (Centers for Disease Control and Prevention, 2023). With the CDC's (2023) recommendation of around 1,300 mg of sodium per meal, a single fast-food entrée can easily exhaust that entire allowance. This raises a straightforward but important prediction question: **Can we predict whether a fast food item is high or low in sodium based on its macronutrient composition?**

This project investigates this research question using nutritional data gathered by Arel-Bundock (2024), comprising 515 observations and 17 variables across eight major U.S. fast-food chains. Each observation represents a single menu item; key variables include sodium, calories, total fat, total carbohydrates, protein, fiber, restaurant chain, and item name. A binary outcome variable, `high_sodium`, is constructed to indicate whether an item exceeds the relevant sodium threshold.

The primary goal is **prediction**: we aim to build a binary classification model using logistic regression that accurately classifies items as high or low sodium from macronutrient information alone. Because salad items have a distinct nutritional profile and a different dietary sodium benchmark, separate models are fitted for salad and non-salad entrées. This model could help consumers make lower-sodium choices and give policymakers a clearer picture of where sodium risk is concentrated across fast-food menus.

## (3) Method and Results

## Data Summary

### Data Source and Collection

The dataset is distributed via the [`openintro`](https://cran.r-project.org/package=openintro) R package and was compiled by Arel-Bundock (2024). The data cover eight major U.S. fast-food chains: McDonald's, Burger King, Taco Bell, Sonic, Arby's, Dairy Queen, Chick-fil-A, and Subway.

The collection method is not explicitly documented in the dataset. Based on its structure and content, the most plausible assumption is that nutritional values were scraped or manually compiled from each restaurant's publicly available nutrition information (e.g., official websites or printed nutrition guides), which are required disclosures under U.S. FDA menu-labelling regulations. Under this assumption:

- Each observation corresponds to one menu item as listed by the restaurant at a point in time.
- Values reflect standard/default preparation with no customization.
- The data represent a cross-sectional snapshot; menu compositions and nutritional content can change over time.
- Items may not be nationally uniform, as regional menu variations may not be captured.

### Variable Descriptions

The dataset contains 515 observations and 17 variables. Key variables used in this analysis are:

| Variable | Type | Description |
|---|---|---|
| `restaurant` | Categorical | Fast-food chain name (8 chains) |
| `item` | Categorical | Menu item name |
| `calories` | Continuous | Total calories (kcal) |
| `total_fat` | Continuous | Total fat (g) |
| `total_carb` | Continuous | Total carbohydrates (g) |
| `protein` | Continuous | Protein content (g) |
| `fiber` | Continuous | Dietary fiber (g) |
| `sodium` | Continuous | Sodium content (mg) — the basis for our response |
| `salad` | Categorical (derived) | Whether the item is a salad (reconstructed from item name) |
| `high_sodium` | Binary (derived) | **Response variable**: High (≥ 1,300 mg for non-salad; ≥ 800 mg for salad) or Low |

Micronutrient variables (`vit_a`, `vit_c`, `calcium`) have approximately 41% missing values and are excluded from modelling. All macronutrient variables have negligible missingness and are reliable for analysis.

In [2]:
library(scales)
library(tidyverse)
library(dplyr)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.6
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ readr::col_factor() masks scales::col_factor()
✖ purrr::discard()    masks scales::discard()
✖ dplyr::filter()     masks stats::filter()
✖ dplyr::lag()        masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [3]:
#Reproducible dataset in R 
# file_url <- "https://www.openintro.org/data/csv/fastfood.csv"
# dest_file <- "fastfood.csv"
# download.file(file_url, dest_file, method = "curl")

fastfood <- read.csv("https://www.openintro.org/data/csv/fastfood.csv")

cat("Number of observations:", nrow(fastfood), "\n")
cat("Number of variables:   ", ncol(fastfood), "\n")

Number of observations: 515 
Number of variables:    17 


## Exploratory Data Analysis

### Data Cleaning and Wrangling

The `salad` variable in the raw dataset contains only the value `"Other"` despite salad items existing in the data. It was reconstructed by detecting the word "salad" (case-insensitive) in the item name. Both `restaurant` and `salad` were converted to factors, with McDonald's set as the reference level for `restaurant`. The binary response variable `high_sodium` was then constructed using item-type-specific thresholds: ≥ 1,300 mg for non-salad items and ≥ 800 mg for salad items, consistent with CDC (2023) dietary sodium guidelines.

In [4]:
non_salad_na <- 1300
salad_na <- 800

fastfood_clean <- fastfood |>

  # Fix salad: contains only 'Other' despite salad items existing.
  # Reconstruct it by detecting 'salad' in the item name.
  mutate(salad = if_else(str_detect(str_to_lower(item), "salad"), "Salad", "Other")) |>

  # Convert categorical variables to factors + add high_sodium column
  mutate(
    restaurant = factor(restaurant),
    restaurant = relevel(restaurant, ref = "Mcdonalds"),
    salad      = factor(salad, levels = c("Other", "Salad")),
    high_sodium = factor(
      case_when(
        salad == "Salad" & sodium >= salad_na  ~ "High",
        salad == "Other" & sodium >= non_salad_na ~ "High",
        TRUE                             ~ "Low"
      ),
      levels = c("Low", "High")
    )
  )

# tail(fastfood_clean)

head(fastfood_clean)

,restaurant,item,calories,cal_fat,total_fat,sat_fat,trans_fat,cholesterol,sodium,total_carb,fiber,sugar,protein,vit_a,vit_c,calcium,salad,high_sodium
,<fct>,<chr>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<fct>,<fct>
1,Mcdonalds,Artisan Grilled Chicken Sandwich,380,60,7,2,0.0,95,1110,44,3,11,37,4,20,20,Other,Low
2,Mcdonalds,Single Bacon Smokehouse Burger,840,410,45,17,1.5,130,1580,62,2,18,46,6,20,20,Other,High
3,Mcdonalds,Double Bacon Smokehouse Burger,1130,600,67,27,3.0,220,1920,63,3,18,70,10,20,50,Other,High
4,Mcdonalds,Grilled Bacon Smokehouse Chicken Sandwich,750,280,31,10,0.5,155,1940,62,2,18,55,6,25,20,Other,High
5,Mcdonalds,Crispy Bacon Smokehouse Chicken Sandwich,920,410,45,12,0.5,120,1980,81,4,18,46,6,20,20,Other,High
6,Mcdonalds,Big Mac,540,250,28,10,1.0,80,950,46,3,9,25,10,2,15,Other,Low


### Missing Values

Three variables — `vit_a`, `vit_c`, and `calcium`, have approximately 41-41.6% missing values, well above the 20% threshold. This likely reflects inconsistent reporting practices across chains rather than data-entry errors; some chains may not have disclosed micronutrient percentages. These variables are excluded from modelling. All macronutrient variables (`calories`, `total_fat`, `sodium`, `protein`, etc.) have negligible or zero missingness and are reliable for analysis.

In [9]:
na_counts <- colSums(is.na(fastfood_clean))
na_pct    <- round(na_counts / nrow(fastfood_clean) * 100, 1)
data.frame(n_missing = na_counts, pct_missing = na_pct) |>
    arrange(desc(pct_missing))

,n_missing,pct_missing
,<dbl>,<dbl>
vit_a,214,41.6
vit_c,210,40.8
calcium,210,40.8
fiber,12,2.3
protein,1,0.2
restaurant,0,0.0
item,0,0.0
calories,0,0.0
cal_fat,0,0.0


### Class Distribution

Two sources of imbalance are present in the categorical variables:

1. **`restaurant`**: Taco Bell (22.3%) and Subway (18.6%) together account for over 40% of all items, while Chick-fil-A contributes only 27 items (5.2%). This means per-chain estimates will be more precise for larger chains, though the imbalance is moderate and manageable in a regression context.

2. **`salad`**: Only 64 of 515 items (12.4%) are salads. This matters because salads tend to have atypical sodium profiles — high sodium from dressings despite lower calorie counts — and their underrepresentation could mask or distort sodium patterns if not accounted for explicitly. This reinforces the decision to fit separate stratified models.

After filtering to complete cases and stratifying by item type, the final analysis datasets contain 261 non-salad items and 40 salad items.

In [10]:
df_nonsalad <- fastfood_clean |>
  filter(salad == "Other") |>
  drop_na()

df_salad <- fastfood_clean |>
  filter(salad == "Salad") |>
  drop_na()

cat("Non-salad items (complete cases):", nrow(df_nonsalad), "\n")
cat("Salad items (complete cases):    ", nrow(df_salad),    "\n")

# ── Class balance check ───────────────────────────────────────────────────
cat("\nNon-salad high_sodium split:\n")
print(table(df_nonsalad$high_sodium))

cat("\nSalad high_sodium split:\n")
print(table(df_salad$high_sodium))

Non-salad items (complete cases): 261 
Salad items (complete cases):     40 

Non-salad high_sodium split:

 Low High 
 149  112 

Salad high_sodium split:

 Low High 
  19   21 


### Visualization: Sodium vs. Calories Across Fast-Food Chains

**Figure 1** displays the relationship between calories and sodium for non-salad and salad items separately, with per-chain linear regression lines and point size scaled to total fat content.

Two key patterns emerge:

- **Diverging slopes (non-salad panel):** The per-chain regression lines diverge noticeably rather than running in parallel, supporting the presence of a `calories × restaurant` interaction. McDonald's stands out most dramatically, with a considerably steeper slope than all other chains — sodium rises much faster with calories for McDonald's items than for comparable items at other restaurants. Arby's and Burger King also show relatively steep slopes, while Subway, Taco Bell, and Sonic show flatter lines.

- **Salad items behave differently:** Salads exhibit a notably different pattern. Sodium levels are generally lower and the calorie range is narrower (roughly 200–800 kcal), yet some chains still show steep sodium increases with calories, likely driven by high-calorie, high-sodium dressings. This confirms that fitting a separate salad model is more appropriate than pooling item types, as pooling would obscure the underlying macronutrient–sodium relationship.

In [7]:
#Add EDA ggplot code cell here

## Methods

Proposed Method
Binary logistic regression is proposed to address this question. Two models are fitted independently: one for non-salad items and one for salad items. A LASSO-based variable selection step is applied within each stratum before the final model is fit.

Why logistic regression?

The response variable high_sodium is binary (High / Low), which violates the normality and continuous-outcome assumptions of linear regression. Logistic regression directly models the log-odds of an item being classified as high sodium as a linear function of the predictors, and produces interpretable odds ratios after exponentiation. It does not require the predictors to be normally distributed, making it well-suited to nutritional data that is typically right-skewed.

Why two separate models?

The high-sodium threshold differs by item type (1,300 mg for non-salad items versus 800 mg for salad items), and the macronutrient composition of salad items is structurally different from non-salad items. Fitting a single pooled model with salad as a covariate would impose the same predictor-outcome relationships across both groups, which is inappropriate given the different classification boundaries. Stratified models allow the coefficients to differ freely between strata.

Why LASSO for variable selection?

Although only four candidate predictors are considered (total_fat, total_carb, protein, fiber), they are moderately correlated because all macronutrients increase with item size. LASSO penalizes the sum of the absolute values of the regression coefficients, shrinking some coefficients exactly to zero and thereby removing those predictors from the model. This produces a sparse, interpretable final model that retains only the macronutrients carrying the most independent information about the outcome. The regularization parameter lambda is selected by 10-fold cross-validation on the training data, minimizing binomial deviance. Predictors with nonzero coefficients at the chosen lambda are passed to the final logistic regression. Note that calories is excluded from the candidate set entirely because it is approximately a linear combination of fat, carbohydrates, and protein (calories = 9 * fat + 4 * carbs + 4 * protein), which would introduce near-perfect multicollinearity.

Data splitting

Each stratum is partitioned into a training set (80%) and a test set (20%) before any modeling begins. All variable selection, cross-validation, and model fitting are performed on the training data only. The test set is used exclusively for the final assessment of predictive performance, ensuring that reported metrics (accuracy, AUC) reflect performance on unseen data.

Goal: prediction and inference

From a prediction standpoint, the models are assessed on how accurately they classify unseen items as high or low sodium. From an inference standpoint, the statistically significant odds ratios in the final logistic regression reveal which macronutrients are most strongly associated with high sodium content, and in which direction.



In [ ]:
#Add LASSO and final model code 

## 3d) Code
## LASSO logistic regression 



ERROR: Error in library(glmnet): there is no package called ‘glmnet’


In [ ]:
#LASSO for the non-salad model 
y_nonsalad <- ifelse(df_nonsalad$high_sodium == "High", 1, 0)

X_nonsalad <- model.matrix(
  high_sodium ~ total_fat + total_carb + protein + fiber,
  data = df_nonsalad
)[, -1]

cv_lasso_nonsalad <- cv.glmnet(
  X_nonsalad, y_nonsalad,
  family = "binomial",
  alpha = 1
)

coef_nonsalad <- coef(cv_lasso_nonsalad, s = "lambda.min")

selected_nonsalad <- rownames(coef_nonsalad)[as.vector(coef_nonsalad != 0)]

selected_nonsalad <- setdiff(selected_nonsalad, "(Intercept)")
print(selected_nonsalad)

ERROR: Error in eval(expr, envir, enclos): object 'df_nonsalad' not found


The non salad model selected variables that are strongly associated with the total sodium content. Our final model for non-salad will include "total_fat", "total_carb", "protein" as the predictor variables. This intuitively makes sense as foods rich in carbohydrates and fat are prepared with processed sauces and fried food (salad dressings) which have high sodium. Processed meat is high in sodium content as well.

In [ ]:
plot(cv_lasso_nonsalad)
title(sub = "Non-Salad: LASSO CV Curve")

The CV LASSO curve for the non-salad model shows that a moderate level of regularization minimizes the cross-validated prediction error. The optimal penalty parameter, λ_min corresponds to the lowest point on the curve and selects a parsimonious set of predictors.At this value of λ, the LASSO model retains three key variables: total_fat, total_carb, and protein.

In [ ]:
#Model used in non-salad model 
final_nonsalad <- glm(
  high_sodium ~ total_fat + total_carb + protein,
  data = df_nonsalad,
  family = binomial
)

summary(final_nonsalad)
exp(coef(final_nonsalad))

At α = 0.05 total_carb and protein are statistically significant predictors in the model, as their p-values are less than 0.05. In contrast, total_fat is not statistically significant at this level 
(p=0.07), indicating weaker evidence of association with the response.

In [ ]:
#lasso for salad model 
y_salad <- ifelse(df_salad$high_sodium == "High", 1, 0)

X_salad <- model.matrix(
  high_sodium ~ total_fat + total_carb + protein + fiber,
  data = df_salad
)[, -1]

cv_lasso_salad <- cv.glmnet(
  X_salad, y_salad,
  family = "binomial",
  alpha = 1
)
coef_salad <- coef(cv_lasso_salad, s = "lambda.min")
coef_salad

selected_salad <- rownames(coef_salad)[as.vector(coef_salad != 0)]
selected_salad <- setdiff(selected_salad, "(Intercept)")

print(selected_salad)

For the salad model,LASSO selected total_fat and protein as the predictors most strongly associated with sodium content.This selection is intuitive. Total fat reflects the use of dressings, oils, and toppings, which are often high in sodium. Protein captures components such as processed or seasoned meats and protein-rich toppings that contain added salt.

In [ ]:
plot(cv_lasso_salad)
title(sub = "Non-Salad: LASSO CV Curve")

The CV LASSO curve for the salad model shows that a moderate level of regularization minimizes the cross-validated prediction error. The optimal penalty parameter, λ_min gives a model that retains three variables: total_fat, protein, and fiber.

In [ ]:
#model to use 
final_salad <- glm(
  high_sodium ~ total_fat + protein,
  data = df_salad,
  family = binomial
)


summary(final_salad)
exp(coef(final_salad))

At α = 0.05 total_fat is a statistically significant predictor in the model as its p-value is less than 0.05. In contrast, protein is not statistically significant at this level (p=0.06843), indicating weaker evidence of association with the response.

## Results

### LASSO Variable Selection

LASSO logistic regression (λ chosen by 10-fold cross-validation) selected the following predictors:

- **Non-salad model:** `total_fat`, `total_carb`, `protein`
- **Salad model:** `total_fat`, `protein`

`fiber` was not retained in either model after regularization, indicating it carries limited independent information about sodium classification once the other macronutrients are included.

---

### Final Logistic Regression Models

#### Non-salad model

$$\log\left(\frac{p}{1-p}\right) = -6.286 + 0.027(\text{total\_fat}) + 0.058(\text{total\_carb}) + 0.075(\text{protein})$$

#### Salad model

$$\log\left(\frac{p}{1-p}\right) = -4.704 + 0.234(\text{total\_fat}) + 0.104(\text{protein})$$

where $p$ is the probability that an item is classified as high sodium.

---

### Coefficient Interpretation

#### Non-salad model

| Predictor | Coefficient | Odds Ratio | p-value | Significant (α = 0.05)? |
|---|---|---|---|---|
| Intercept | −6.286 | 0.0019 | — | — |
| `total_fat` | 0.027 | 1.028 | 0.072 | No |
| `total_carb` | 0.058 | 1.059 | < 0.001 | **Yes** |
| `protein` | 0.075 | 1.078 | < 0.001 | **Yes** |

- Each additional gram of **total carbohydrates** is associated with a 5.9% increase in the odds of being classified as high sodium, holding other predictors constant (OR = 1.059, p < 0.001).
- Each additional gram of **protein** is associated with a 7.8% increase in the odds of being classified as high sodium (OR = 1.078, p < 0.001).
- **Total fat** was retained by LASSO but is not statistically significant at the 5% level (p = 0.072).

#### Salad model

| Predictor | Coefficient | Odds Ratio | p-value | Significant (α = 0.05)? |
|---|---|---|---|---|
| Intercept | −4.704 | 0.0091 | — | — |
| `total_fat` | 0.234 | 1.264 | 0.008 | **Yes** |
| `protein` | 0.104 | 1.109 | 0.068 | Marginal |

- Each additional gram of **total fat** is associated with a 26.4% increase in the odds of a salad item being classified as high sodium (OR = 1.264, p = 0.008).
- **Protein** shows a marginal association with a 10.9% increase in odds per gram (OR = 1.109, p = 0.068), but does not reach significance at the 5% level.

---

### Model Fit

| | Non-salad model | Salad model |
|---|---|---|
| Null deviance | 356.56 (df = 260) | 55.35 (df = 39) |
| Residual deviance | 203.66 (df = 257) | 27.98 (df = 37) |
| AIC | 211.66 | 33.98 |

Both models show a meaningful reduction in deviance relative to the null (intercept-only) model, indicating that macronutrient composition carries real predictive information about sodium classification.

---

### Visualization: Odds Ratios

**Figure 2** displays the odds ratios and 95% confidence intervals for each predictor in both models on a log scale. Points coloured to indicate statistical significance (p < 0.05). In the non-salad model, `total_carb` and `protein` have narrow intervals clearly above the OR = 1 reference line, while `total_fat`'s interval straddles 1. In the salad model, `total_fat` stands well above 1 with a tighter interval, while `protein` spans a wider range reflecting greater uncertainty. This divergence in dominant predictors between models visually reinforces the appropriateness of fitting separate models.

In [ ]:
#Add odds-ratio ggplot code cell here

## Discussion

### Summary of Findings

This project investigated whether fast-food menu items could be classified as high- or low-sodium based solely on their macronutrient composition data. Using binary logistic regression with LASSO-based variable selection, two separate models were fitted: one for non-salad items and one for salad items with total fat, total carbohydrates, protein, and fiber as predictor variables.

For non-salad items, LASSO selected total fat, total carbohydrates, and protein. The final logistic regression model showed a substantial reduction in deviance from the null model (356.56 to 203.66), with total carbohydrates (OR = 1.059, p < 0.001) and protein (OR = 1.078, p < 0.001) both statistically significant at the 5% level. 
[NOTE: OR here stands for Odds-Ratio.] 

Total fat was retained by LASSO but was not statistically significant in the final model (p = 0.072). For salad items, LASSO selected only total fat and protein. The final regression model again showed meaningful improvement over the null (deviance reduced from 55.35 to 27.98), with total fat statistically significant (OR = 1.264, p = 0.008) and protein marginally so (OR = 1.109, p = 0.068).

These findings support our hypothesis that macronutrient composition carries meaningful information about whether a fast food item is high in sodium. 

Protein emerged as a statistically significant predictor in both models, consistent with the high sodium load of processed meats and seasoned proteins common with fast food menus. 

Total carbohydrates were the strongest predictor for non-salad items, likely reflecting through buns, breads, and sauces. 
Total fat was the dominant predictor for salad items, which aligns with the role of sodium-heavy dressings, cheeses, and processed toppings. 

The fact that different macronutrients drive high sodium classification in each model supports the decision to fit separate models rather than a single pooled one, as the relationship between macronutrient composition and sodium risk is not uniform across item types.

### Assumptions and Limitations

Several important assumptions and limitations should be considered. Binary logistic regression assumes a linear relationship between each predictor and the log-odds of the outcome. Residual plots could be used to diagnose potential non-linearity.

The independence assumption may also be violated. Menu items from the same restaurant share recipes, preparation practices, and seasoning conventions, meaning residuals within a chain are likely correlated. This was not accounted for in either model, which could lead to underestimated standard errors and overstated significance for some predictors.

The salad model was fitted on only 40 complete cases, which is a small sample for logistic regression with multiple predictors, meaning estimates carry substantially more uncertainty and should be interpreted cautiously.

Finally, the binary sodium thresholds (1,300 mg for non-salad items; 800 mg for salad items) were fixed based on CDC dietary guidelines rather than derived from the data. The choice of threshold directly shapes the class balance and model outcomes. Different cutoffs would yield different classifications and potentially different conclusions.


### Potential Model Improvements

Several improvements could strengthen the analysis. Including restaurant chain as a predictor or fitting chain-stratified models could account for the systematic sodium differences across chains identified in the EDA, where chains like McDonald's showed notably steeper calorie-sodium relationships than others. Formal cross-validation or a held-out test set would enable a more rigorous evaluation of predictive accuracy beyond deviance-based model fit. The salad model in particular would benefit from a larger sample, either by collecting additional data or by pooling across a broader set of restaurant chains and menu databases. Additionally, non-linear terms or interaction effects between macronutrients could be explored to better capture the complexity of sodium in highly processed foods.

### Future Directions

Future work could extend our analysis in several directions. We could incorporate restaurant-level effects explicitly, either through interaction terms or multilevel models, to account for within-chain similarities. Another direction would be to include more detailed ingredient-level data, which could provide a more direct link between specific components (e.g., sauces, cheeses) and sodium content. Expanding the dataset to include more observations, particularly for underrepresented categories such as salads, would also improve model stability.

Beyond methodological improvements, this study raises broader questions about nutritional transparency and public health. For example, future research could investigate whether certain restaurant chains systematically exceed sodium guidelines even after controlling for nutritional composition, or whether menu labeling policies influence consumer choices. Additionally, exploring how sodium content interacts with other health metrics, such as calories or fat intake, could provide a more comprehensive view of dietary risk.

## References

Arel-Bundock, V. (2024). *Rdatasets: A collection of datasets distributed with R* (Dataset: fastfood) [Data repository]. https://vincentarelbundock.github.io/Rdatasets/

Centers for Disease Control and Prevention. (2023, September 14). *Sodium reduction: State policy interventions by evidence level.* CDC Archive. https://archive.cdc.gov/www_cdc_gov/dhdsp/policy_resources/sodium/evidence_interventions.htm

Gardener, H., Rundek, T., Wright, C. B., Elkind, M. S. V., & Sacco, R. L. (2012). Dietary sodium and risk of stroke in the Northern Manhattan Study. *Stroke, 43*(5), 1200–1205. https://doi.org/10.1161/STROKEAHA.111.641043

Shah, N. N., Fryar, C. D., Ahluwalia, N., & Akinbami, L. J. (2025, June). *Fast-food intake among adults in the United States, August 2021–August 2023* (NCHS Data Brief No. 533). National Center for Health Statistics. https://doi.org/10.15620/cdc/174606

U.S. Food and Drug Administration. (2024, March 5). *Sodium in your diet.* https://www.fda.gov/food/nutrition-education-resources-materials/sodium-your-diet

Wang, Y. J., Yeh, T. L., Shih, M. C., Tu, Y. K., & Chien, K. L. (2020). Dietary sodium intake and risk of cardiovascular disease: A systematic review and dose-response meta-analysis. *Nutrients, 12*(10), 2934. https://doi.org/10.3390/nu12102934

## AI Acknowledgement

ChatGPT (OpenAI, April 2026 version) was used to generate initial APA 7 citation formatting for the references listed above. All citations were reviewed and verified for accuracy by the group. All sources were selected by the group members, and we take full responsibility for the content of this submission.

Claude (Anthropic) was used to suggest improvements to `ggplot` visual formatting and theme adjustments for enhanced readability, and to assist with Markdown cell structure and formatting within the notebook. All analytical decisions, code logic, and written interpretations are our own.